# SMS Spam Detection Using Machine Learning
**Student:** Abhishek

Classify SMS messages as spam or ham using TF-IDF and machine learning. The notebook downloads the public UCI SMS Spam Collection dataset on first run; internet access is required.

In [ ]:
import os, zipfile, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

## 1. Load dataset

In [ ]:
os.makedirs('data', exist_ok=True)
zip_path='data/smsspamcollection.zip'
url='https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
if not os.path.exists(zip_path): urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    name=next(n for n in z.namelist() if n.endswith('SMSSpamCollection'))
    with z.open(name) as f: df=pd.read_csv(f, sep='\t', header=None, names=['label','message'], encoding='utf-8')
df.head()

## 2. Inspect and clean data

In [ ]:
print('Shape:', df.shape)
print(df.isna().sum())
print(df['label'].value_counts())
df=df.dropna(subset=['label','message']).drop_duplicates().copy()
df['label']=df['label'].str.lower().str.strip()
df=df[df.label.isin(['ham','spam'])]
df['target']=df['label'].map({'ham':0,'spam':1})
df.head()

## 3. Exploratory data analysis

In [ ]:
df['label'].value_counts().plot(kind='bar', rot=0)
plt.title('SMS Class Distribution'); plt.xlabel('Label'); plt.ylabel('Messages'); plt.tight_layout(); plt.show()
df['message_length']=df.message.str.len()
df.groupby('label')['message_length'].describe()

## 4. Train and compare models
TF-IDF converts text to numerical features. Compare Multinomial Naive Bayes and Logistic Regression.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df.message,df.target,test_size=0.2,random_state=42,stratify=df.target)
models={
 'Multinomial Naive Bayes':Pipeline([('tfidf',TfidfVectorizer(stop_words='english')),('model',MultinomialNB())]),
 'Logistic Regression':Pipeline([('tfidf',TfidfVectorizer(stop_words='english')),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))])}
results=[]
for name,model in models.items():
 model.fit(X_train,y_train); pred=model.predict(X_test)
 results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1-score':f1_score(y_test,pred,zero_division=0)})
results_df=pd.DataFrame(results).sort_values('F1-score',ascending=False)
results_df

## 5. Evaluation

In [ ]:
best_name=results_df.iloc[0]['Model']; best_model=models[best_name]
pred=best_model.predict(X_test)
print('Selected model:',best_name)
print(classification_report(y_test,pred,target_names=['Ham','Spam'],zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Ham','Spam'])
plt.tight_layout(); plt.show()

## 6. Test sample messages

In [ ]:
samples=['Congratulations! You have won a free prize. Call now to claim.','Hi, are we still meeting at the library today?']
for msg,label in zip(samples,best_model.predict(samples)):
 print(msg,'=>','SPAM' if label==1 else 'HAM (not spam)')

## 7. Conclusion and limitations
This project demonstrates text classification using TF-IDF and two models. Review precision, recall and F1-score as well as accuracy. The model can make mistakes and is an educational demonstration, not a guaranteed spam filter.